In [1]:
from marubatsu import BitBoard

def __init__(self, board_size:int=3, count_linemark:bool=False):
    self.BOARD_SIZE = board_size
    self.bit_length = self.BOARD_SIZE ** 2
    self.count_linemark = count_linemark

    # 参照テーブルの計算
    self.fullmask = (1 << self.BOARD_SIZE ** 2) - 1
    self.colmasks = []
    self.rowmasks = []
    self.diamask1 = 0
    self.diamask2 = 0
    for i in range(self.BOARD_SIZE):
        colmask = 0
        rowmask = 0
        for j in range(self.BOARD_SIZE):
            colmask |= self.xy_to_move(i, j)
            rowmask |= self.xy_to_move(j, i)
        self.colmasks.append(colmask)
        self.rowmasks.append(rowmask)
        self.diamask1 |= self.xy_to_move(i, i)
        self.diamask2 |= self.xy_to_move(i, self.BOARD_SIZE - i - 1)
    self.masklist = self.colmasks + self.rowmasks + [self.diamask1, self.diamask2]
    
    self.fliplr_ds_table = []
    mask = None
    length = self.BOARD_SIZE
    while length > 1:
        delta = (length + 1) // 2 * self.BOARD_SIZE
        length //= 2
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_ds_table.append((delta, mask))
        prevdelta = delta
        
    self.BB_SIZE = 1 << (self.BOARD_SIZE - 1).bit_length()
    self.delta = (self.BB_SIZE - self.BOARD_SIZE) * self.BOARD_SIZE
    self.fliplr_sa_table = []
    mask = None
    length = self.BB_SIZE
    while length > 1:
        length //= 2
        delta = length * self.BOARD_SIZE
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_sa_table.append((delta, mask))
        prevdelta = delta
        
    self.transpose_ds_table = []
    for i in range(1, self.BOARD_SIZE):
        mask = 0
        for j in range(self.BOARD_SIZE - i):
            mask |= self.xy_to_move(j, i + j)
        self.transpose_ds_table.append(((self.BOARD_SIZE - 1) * i, mask))
            
    self.initialize()    
    
BitBoard.__init__ = __init__

In [2]:
def delta_swap(b, mask, delta):
    c = (b ^ (b >> delta)) & mask
    return c ^ (c << delta) ^ b 

def transpose_ds(self):
    for delta, mask in self.transpose_ds_table:
        self.board[0] = delta_swap(self.board[0], mask, delta)
        self.board[1] = delta_swap(self.board[1], mask, delta)

BitBoard.transpose_ds = transpose_ds

In [3]:
from marubatsu import Marubatsu

mb4 = Marubatsu(boardclass=BitBoard, board_size=4)
movelist = [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1), (1, 2), (1, 3), (2, 2), (2, 3), (3, 3)]
for x, y in movelist:
  mb4.cmove(x, y)
print(mb4)
mb4.board.transpose_ds()
print(mb4)
mb4.board.transpose_ds()
print(mb4)

Turn o
o...
xo..
oxx.
xooX

Turn o
oxox
.oxo
..xo
...X

Turn o
o...
xo..
oxx.
xooX



In [4]:
mb8 = Marubatsu(boardclass=BitBoard, board_size=8)
for x in range(8):
    for y in range(x, 8):
        mb8.cmove(x, y)
print(mb8)

mask = 0b0000000000000000000000000000000011110000111100001111000011110000
# mask = 0x00000000f0f0f0f0
delta = 28
mb8.board.board[0] = delta_swap(mb8.board.board[0], mask, delta)
mb8.board.board[1] = delta_swap(mb8.board.board[1], mask, delta)
print(mb8)

Turn o
o.......
xo......
oxx.....
xoox....
oxxoo...
xooxxo..
oxxooxx.
xooxxooX

Turn o
o...oxxo
xo..xoox
oxx.oxxo
xooxxoox
....o...
....xo..
....oxx.
....xooX



In [5]:
print(mb8)
mask = 0b0000000000000000110011001100110000000000000000001100110011001100
# mask = 0x0000cccc0000cccc
delta = 14
mb8.board.board[0] = delta_swap(mb8.board.board[0], mask, delta)
mb8.board.board[1] = delta_swap(mb8.board.board[1], mask, delta)
print(mb8)

Turn o
o...oxxo
xo..xoox
oxx.oxxo
xooxxoox
....o...
....xo..
....oxx.
....xooX

Turn o
o.oxoxox
xoxoxoxo
..x.xoxo
..oxoxox
....o.ox
....xoxo
......x.
......oX



In [6]:
print(mb8)
mask = 0b0000000010101010000000001010101000000000101010100000000010101010
# mask = 0x00aa00aa00aa00aa
delta = 7
mb8.board.board[0] = delta_swap(mb8.board.board[0], mask, delta)
mb8.board.board[1] = delta_swap(mb8.board.board[1], mask, delta)
print(mb8)

Turn o
o.oxoxox
xoxoxoxo
..x.xoxo
..oxoxox
....o.ox
....xoxo
......x.
......oX

Turn o
oxoxoxox
.oxoxoxo
..xoxoxo
...xoxox
....oxox
.....oxo
......xo
.......X



In [7]:
ni = 4
M1 = ((1 << ni) - 1) << ni
print(f"0b{M1:b}")
mb8.restart()
mb8.board.board[0] = M1
print(mb8)

0b11110000
Turn o
........
........
........
........
o.......
o.......
o.......
o.......



In [8]:
n = 8
M2 = M1
w = 1
while w < ni:
    M2 |= M2 << (w * n)
    w *= 2
print(f"0b{M2:b}")
mb8.board.board[0] = M2
print(mb8)

0b11110000111100001111000011110000
Turn o
........
........
........
........
oooo....
oooo....
oooo....
oooo....



In [9]:
n = 8
ni = 2

M1 = ((1 << ni) - 1) << ni
print(f"0b{M1:b}")
mb8.restart()
mb8.board.board[0] = M1
print("M1")
print(mb8)

M2 = M1
w = 1
while w < ni:
    M2 |= M2 << (w * n)
    w *= 2
print(f"0b{M2:b}")
mb8.board.board[0] = M2
print("M2")
print(mb8)

M3 = M2
h = ni * 2
while h < n:
    M3 |= M3 << h
    h *= 2
print(f"0b{M3:b}")
mb8.board.board[0] = M3
print("M3")
print(mb8)

0b1100
M1
Turn o
........
........
o.......
o.......
........
........
........
........

0b110000001100
M2
Turn o
........
........
oo......
oo......
........
........
........
........

0b1100110011001100
M3
Turn o
........
........
oo......
oo......
........
........
oo......
oo......



In [10]:
Mi = M3
w = ni * 2
while w < n:
    Mi |= Mi << (w * n)
    w *= 2
print(f"0b{Mi:b}")
mb8.board.board[0] = Mi
print("Mi")
print(mb8)

0b110011001100110000000000000000001100110011001100
Mi
Turn o
........
........
oo..oo..
oo..oo..
........
........
oo..oo..
oo..oo..



In [11]:
from marubatsu import BitBoard

def __init__(self, board_size:int=3, count_linemark:bool=False):
    self.BOARD_SIZE = board_size
    self.bit_length = self.BOARD_SIZE ** 2
    self.count_linemark = count_linemark

    # 参照テーブルの計算
    self.fullmask = (1 << self.BOARD_SIZE ** 2) - 1
    self.colmasks = []
    self.rowmasks = []
    self.diamask1 = 0
    self.diamask2 = 0
    for i in range(self.BOARD_SIZE):
        colmask = 0
        rowmask = 0
        for j in range(self.BOARD_SIZE):
            colmask |= self.xy_to_move(i, j)
            rowmask |= self.xy_to_move(j, i)
        self.colmasks.append(colmask)
        self.rowmasks.append(rowmask)
        self.diamask1 |= self.xy_to_move(i, i)
        self.diamask2 |= self.xy_to_move(i, self.BOARD_SIZE - i - 1)
    self.masklist = self.colmasks + self.rowmasks + [self.diamask1, self.diamask2]
    
    self.fliplr_ds_table = []
    mask = None
    length = self.BOARD_SIZE
    while length > 1:
        delta = (length + 1) // 2 * self.BOARD_SIZE
        length //= 2
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_ds_table.append((delta, mask))
        prevdelta = delta
        
    self.BB_SIZE = 1 << (self.BOARD_SIZE - 1).bit_length()
    self.delta = (self.BB_SIZE - self.BOARD_SIZE) * self.BOARD_SIZE
    self.fliplr_sa_table = []
    mask = None
    length = self.BB_SIZE
    while length > 1:
        length //= 2
        delta = length * self.BOARD_SIZE
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_sa_table.append((delta, mask))
        prevdelta = delta

    self.transpose_ds_table = []
    for i in range(1, self.BOARD_SIZE):
        mask = 0
        for j in range(self.BOARD_SIZE - i):
            mask |= self.xy_to_move(j, i + j)
        self.transpose_ds_table.append(((self.BOARD_SIZE - 1) * i, mask))
            
    self.transpose_dc_table = []
    n = self.BOARD_SIZE
    ni = n
    while ni > 1:
        ni //= 2
        delta = (n - 1) * ni
        # M1 を計算する
        mask = ((1 << ni) - 1) << ni
        # M2 を計算する
        w = 1
        while w < ni:
            mask |= mask << (w * n)
            w *= 2
        # M3 を計算する
        h = ni * 2
        while h < n:
            mask |= mask << h
            h *= 2
        # mi を計算する
        w = ni * 2
        while w < n:
            mask |= mask << (w * n)
            w *= 2
        self.transpose_dc_table.append((delta, mask))
            
    self.initialize()    
    
BitBoard.__init__ = __init__

In [12]:
def transpose_dc(self):
    for delta, mask in self.transpose_dc_table:
        self.board[0] = delta_swap(self.board[0], mask, delta)
        self.board[1] = delta_swap(self.board[1], mask, delta)

BitBoard.transpose_dc = transpose_dc

In [13]:
mb8 = Marubatsu(boardclass=BitBoard, board_size=8)
for x in range(8):
    for y in range(x, 8):
        mb8.cmove(x, y)
print(mb8)
mb8.board.transpose_dc()
print(mb8)
mb8.board.transpose_dc()
print(mb8)

Turn o
o.......
xo......
oxx.....
xoox....
oxxoo...
xooxxo..
oxxooxx.
xooxxooX

Turn o
oxoxoxox
.oxoxoxo
..xoxoxo
...xoxox
....oxox
.....oxo
......xo
.......X

Turn o
o.......
xo......
oxx.....
xoox....
oxxoo...
xooxxo..
oxxooxx.
xooxxooX



In [14]:
for board_size in [4, 8, 16, 32, 64, 128, 256]:
    mb = Marubatsu(boardclass=BitBoard, board_size=board_size)
    print(f"board_size = {board_size}")
    for x in range(board_size):
        for y in range(x, board_size):
            mb.cmove(x, y)
    %timeit mb.board.transpose_ds()
    %timeit mb.board.transpose_dc()


board_size = 4
1.6 μs ± 13.6 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
1.12 μs ± 39.9 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
board_size = 8
4.62 μs ± 45.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.01 μs ± 35 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
board_size = 16
10.5 μs ± 56.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.93 μs ± 57.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
board_size = 32
27.9 μs ± 266 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
4.36 μs ± 11.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
board_size = 64
106 μs ± 399 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
11.9 μs ± 16 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
board_size = 128
416 μs ± 1.66 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
30.3 μs ± 83.1 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
boa